# Step 2: Sionna RT simulation

Load a scene exported by step 1, place transmitters/receivers, compute
propagation paths, and derive CIR/CFR and radio maps from them.

**Provenance:** adapted from real, previously-run notebooks (not written
from scratch), mainly a CityGML/38-building scene run and an
IFC/Gleishalle WiFi-style run, both of which executed successfully with
real output shapes printed. Where this notebook simplifies or generalizes
something scene-specific from those runs (transmitter/receiver positions,
antenna configs), that's called out explicitly rather than presented as
universal defaults.

**What this notebook does NOT do yet:** the root README and this folder's
own README describe step 2 as running "parameter sweeps (positions, counts,
materials)". Nothing in the reference material this was built from actually
implements a sweep. Every prior run is a single hand-picked configuration.
This notebook covers that single-configuration case end to end; turning it
into an actual sweep (loop over configs, collect/organize results, respect
the repo-size guidance in the root README) is still open work, not done
here.

## 0. Before you start

- Needs `sionna-rt` (see the root README's Setup table / `requirements.txt`
  at the repo root). `pip install sionna-rt` also pulls in a compatible
  `mitsuba`/`drjit`.
- **Mitsuba variant must end in `_ad_mono_polarized`**, not `_ad_rgb`. This
  matches the fix already made in `1a`'s verification section, per the
  [official Sionna RT tutorial](https://nvlabs.github.io/sionna/rt/tutorials/Introduction.html).
  Keep step 1 and step 2 using the same variant family or results aren't
  comparable.
- **Scene source: both are supported, pick one.** The root README documents
  loading `scenes/<name>/<name>.xml` from a local git checkout, but in
  practice this has been run on Google Colab against a Google
  Drive-mounted path instead. `SCENE_SOURCE` below switches between them.
  If you're on Colab, prefer cloning this repo (with `git lfs pull`) over a
  hand-copied Drive folder, so the scene data doesn't silently drift out of
  sync with what's in git. See the root README's Setup section.
- **Windows laptop with both an integrated and NVIDIA GPU, and Mitsuba
  won't pick up CUDA?** Set these two environment variables *before*
  importing `mitsuba`/`sionna.rt` (not after; the CUDA context is
  initialized on import):
  ```python
  import os
  os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
  os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # adjust the index if you have multiple GPUs
  ```
  Both are standard CUDA variables (not Sionna/Dr.Jit-specific) that force
  the dedicated GPU to be selected instead of the integrated one. One thing
  *not* included here: a previous version of this troubleshooting also set
  `DRJIT_FORCE_CUDA=1`. Checked against Dr.Jit's actual reference docs and
  source, and this doesn't appear to be a real, recognized variable. Setting
  an environment variable a library doesn't read is a harmless no-op, but
  it's not a verified fix either, so it's left out here rather than carried
  forward as if it were confirmed.

In [ ]:
# Import or install Sionna
try:
    import sionna.rt
except ImportError:
    import os
    os.system("pip install sionna-rt")
    import sionna.rt

import os
import mitsuba as mi
mi.set_variant('cuda_ad_mono_polarized')  # CPU fallback: 'llvm_ad_mono_polarized'

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from sionna.rt import (load_scene, PlanarArray, Transmitter, Receiver, Camera,
                        PathSolver, RadioMapSolver, subcarrier_frequencies)

no_preview = True  # False uses the interactive preview widget (Jupyter only)

print(f"Mitsuba variant: {mi.variant()}")

## 1. Load the scene

Set `SCENE_SOURCE` to match how you're running this. Either way,
`SCENE_XML_PATH` ends up pointing at the `.xml` a step 1 notebook exported
(see the root README's "Getting a scene..." section for that handoff).

In [ ]:
SCENE_SOURCE = "local"  # "local" or "colab_drive"

if SCENE_SOURCE == "colab_drive":
    from google.colab import drive
    drive.mount('/content/drive')
    SCENE_XML_PATH = "/content/drive/MyDrive/sionna rt/<folder>/<name>.xml"  # <-- edit
elif SCENE_SOURCE == "local":
    # Path into this repo's scenes/ folder. See scenes/README.md for the
    # <name>/<name>.xml layout step 1 writes.
    SCENE_XML_PATH = os.path.join("..", "scenes", "<name>", "<name>.xml")  # <-- edit
else:
    raise ValueError(f"Unknown SCENE_SOURCE: {SCENE_SOURCE!r}")

# Same defensive checks as 1a's verification section. An empty or
# half-written XML otherwise fails with a cryptic ParseError that doesn't
# obviously point back at "the export didn't finish" or "wrong path".
if not os.path.isfile(SCENE_XML_PATH):
    raise FileNotFoundError(f"Scene XML not found: {SCENE_XML_PATH}")
if os.path.getsize(SCENE_XML_PATH) == 0:
    raise ValueError(f"Scene XML is empty, the export likely didn't finish: {SCENE_XML_PATH}")

scene = load_scene(SCENE_XML_PATH)
print(f"Loaded OK. Object count: {len(scene.objects)}")
print("Object names:", list(scene.objects.keys())[:10], "..." if len(scene.objects) > 10 else "")

## 2. Quick look at the scene

Pick a camera position and render a static image (or use `scene.preview()`
in an actual Jupyter session by setting `no_preview = False` above). Purely
a sanity check that the right scene loaded and looks as expected before
placing transmitters/receivers.

In [ ]:
my_cam = Camera(position=[0, 0, 200], look_at=[0, 0, 0])  # <-- edit for your scene's scale/location

if not no_preview:
    scene.preview()
else:
    scene.render(camera=my_cam, resolution=[650, 500], num_samples=256);

## 3. Configure antenna arrays and place transmitters/receivers

**The array config and positions below are examples, not defaults.** Copy
this cell and adapt it per scene, the same way `1a`'s material `MAPPING` or
`1b`'s `prefix_mapping` are meant to be edited per dataset, not used as-is.

Two patterns seen in practice, worth knowing about when adapting this:
- **Single cellular-style tx/rx pair** (shown below): `tr38901` pattern for
  the transmitter, a single receiver, `tx.look_at(rx)`.
- **One AP + multiple receivers** (WiFi-style, e.g. simulating several
  clients around one access point). Same idea, just call `scene.add()`
  for each additional `Receiver`, and set `scene.frequency` explicitly if
  you're not simulating the default cellular band (the WiFi reference run
  this was adapted from used `scene.frequency = 5e9` for a 5 GHz link).

In [ ]:
# Antenna arrays, shared by every transmitter and every receiver respectively.
scene.tx_array = PlanarArray(
    num_rows=1,
    num_cols=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="tr38901",
    polarization="V"
)

scene.rx_array = PlanarArray(
    num_rows=1,
    num_cols=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="tr38901",
    polarization="V"
)

# Example positions. Replace with real coordinates for your scene (e.g.
# picked from the FBX reference file per 1a Section 1.7's workflow).
tx = Transmitter(name="tx1", position=[30, 70, 35])
scene.add(tx)

rx = Receiver(name="rx1", position=[116, 87, 7])
scene.add(rx)

tx.look_at(rx)

## 4. Compute propagation paths

`max_depth` caps the number of ray/object interactions (0 = LOS only).
`seed` makes the diffusely-reflected-path sampling reproducible across runs.
Keep it fixed and recorded if you need to compare results later, since
without it, results can vary run to run even for an identical scene/config.

In [ ]:
p_solver = PathSolver()

paths = p_solver(
    scene=scene,
    max_depth=5,
    los=True,
    specular_reflection=True,
    diffuse_reflection=False,
    refraction=True,
    synthetic_array=False,
    seed=41,
)

if no_preview:
    scene.render(camera=my_cam, paths=paths, clip_at=40);
else:
    scene.preview(paths=paths, clip_at=40);

## 5. From paths to CIR / CFR / taps

Standard Sionna RT conversions. See the
[official tutorial](https://nvlabs.github.io/sionna/rt/tutorials/Introduction.html)
for what each shape dimension means (`[num_rx, num_rx_ant, num_tx,
num_tx_ant, num_paths, num_time_steps]` for the CIR).

In [ ]:
a, tau = paths.cir(normalize_delays=True, out_type="numpy")
print("Shape of a:", a.shape)
print("Shape of tau:", tau.shape)

num_subcarriers = 1024
subcarrier_spacing = 30e3
frequencies = subcarrier_frequencies(num_subcarriers, subcarrier_spacing)

h_freq = paths.cfr(frequencies=frequencies, normalize=True, normalize_delays=True, out_type="numpy")
print("Shape of h_freq:", h_freq.shape)

plt.figure()
plt.plot(np.abs(h_freq)[0, 0, 0, 0, 0, :])
plt.xlabel("Subcarrier index")
plt.ylabel(r"|$h_\text{freq}$|")
plt.title("Channel frequency response");

## 6. Radio map

Assigns a metric (path gain / RSS / SINR) to every point on a plane for
each transmitter in the scene. `cell_size` and `samples_per_tx` trade off
resolution/accuracy against runtime. This is usually the slowest step and
the one most worth checking against the repo-size guidance in the root
README before generating many of these.

In [ ]:
rm_solver = RadioMapSolver()

rm = rm_solver(
    scene=scene,
    max_depth=5,
    cell_size=[1, 1],
    samples_per_tx=10**6,
)

if no_preview:
    scene.render(camera=my_cam, radio_map=rm);
else:
    scene.preview(radio_map=rm);

## Next: turning this into a sweep

This notebook computes one configuration. A real sweep (varying tx/rx
positions, counts, or materials across many runs and collecting the
results) isn't implemented anywhere this was built from. It's open work.
When it gets built, keep the root README's repo-size guidance in mind:
commit code and a few representative results, not every sweep output.